In [ ]:
import torch

x = torch.tensor([[0.0189, 0.0169, 0.0074, 0.0187, 0.0546, 0.0195, 0.0429, 0.0170,
					   0.0188, 0.1521, 0.1634, 0.0626, 0.3049, 0.0444, 0.0219, 0.0360]])


x = [50000, -22200]

y = torch.tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.]])

# calculate the cross entropy loss
loss = torch.nn.CrossEntropyLoss()
output = loss(x, y)
print(output)

x = torch.tensor([[[0.0189, 0.0169, 0.0074, 0.0187, 0.0546, 0.0195, 0.0429, 0.0170,
					   0.0188, 0.1521, 0.1634, 0.0626, 0.3049, 0.0444, 0.0219, 0.0360]]])

y = torch.tensor([[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.]]])

# calculate the cross entropy loss
loss = torch.nn.CrossEntropyLoss()
output = loss(x, y)
print(output)

x = torch.tensor([0.0189, 0.0169, 0.0074, 0.0187, 0.0546, 0.0195, 0.0429, 0.0170,
					   0.0188, 0.1521, 0.1634, 0.0626, 0.3049, 0.0444, 0.0219, 0.0360])

y = torch.tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.])

# calculate the cross entropy loss
loss = torch.nn.CrossEntropyLoss()
output = loss(x, y)
print(output)

tensor(2.5333)
tensor(-0.)
tensor(2.5333)


In [26]:
import torch
import torch.nn as nn
from torch import optim
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [27]:
# get device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [28]:
df_raw = pd.read_csv('/data1/tom/apps/LLMsForTimeSeries/PAttn_lstm/datasets/simple/simple.csv', nrows=50000)
# df_raw = pd.read_csv('/data1/tom/apps/LLMsForTimeSeries/PAttn_lstm/datasets/mcf_50_filtered/mcf_50_filtered.csv', nrows=100000)
cols = list(df_raw.columns)
if 'date' in cols:
    cols.remove('date')
# remove the date column for df_raw
df_raw = df_raw[cols]

label_encoders = [LabelEncoder() for _ in cols]
# for each column, fit the encoder
for i, col in enumerate(cols):
    label_encoders[i].fit(df_raw[col].values)
    df_raw[col] = label_encoders[i].transform(df_raw[col].values)
    print(f'Number of unique values in {col}: {len(label_encoders[i].classes_)}')

Number of unique values in pc: 128
Number of unique values in delta_in: 64
Number of unique values in delta_out: 16


In [ ]:
from torch.nn import LSTM

class PAttn(nn.Module):
    """
    pattn PAttn 
    
    Decomposition-Linear
    """
    def __init__(self, configs, device):
        super(PAttn, self).__init__()
        self.seq_len = configs.seq_len
        self.pred_len = configs.pred_len
        self.patch_size = configs.patch_size 
        self.stride = configs.patch_size //2 
        
        self.d_model = configs.d_model
        self.method = configs.method
        # since classification, need to embed the data
        self.features = configs.features
        self.label_encoders = configs.label_encoders
        assert len(self.label_encoders) == self.features + 1
        # self.embeddings = nn.ModuleList([torch.nn.Embedding(configs.seq_len, self.d_model) for le in (self.label_encoders)])
        self.in_layer = nn.Linear(self.features * self.seq_len, self.d_model)
       
        # self.basic_attn = MultiheadAttention(embed_dim =self.d_model, num_heads=8)
        self.lstm = LSTM(input_size=self.d_model, hidden_size=self.d_model, num_layers=2, batch_first=True)
        
        
        output_embed_dim = len(self.label_encoders[-1].classes_)
        self.out_layer = nn.Linear(self.d_model, configs.pred_len * output_embed_dim)
        
        # a list of FC layers
        self.fc_layers = nn.ModuleList([nn.Linear(self.d_model, self.d_model) for _ in range(5)])
        
        # softmax for each pred_len
        self.softmax = nn.Softmax(dim=2)
        # self.sigmoid = nn.Sigmoid()
        
    # Calculate temperature dynamically for each [i][j] along the feature_dim (dim=2)
    def calculate_temperatures(self, logits, scaling_factor=1.0):
        # Calculate standard deviation along the feature dimension (dim=2)
        std_dev = torch.std(logits, dim=2, keepdim=True)  # Keep dim for broadcasting
        temperature = scaling_factor * std_dev  # Scale temperature
        return temperature
        
    def forward(self, x):
        batch, features, seq_len = x.size()
        # embed the data
        # x = torch.cat([self.embeddings[i](x[:, i, :]) for i in range(features)], dim=1)
        x = x.flatten(start_dim=1).float()
        x = self.in_layer(x)
        # now shape is [batch, features * history, d_model]

        # x, _ = self.basic_attn( x ,x ,x )
        # for fc_layer in self.fc_layers:
        #     x = fc_layer(x)
        x, _ = self.lstm(x)
        # flatten the data, but keep the batch dimension
        x = x.flatten(start_dim=1)
        
        # now shape is [batch, features * history, d_model] want [batch, pred_len, output_embed_dim]
        x = self.out_layer(x)
        
        x = x.reshape(batch, self.pred_len, -1)
        
        # calculate a temprature so that the values are not too extreme
        # temprature = self.calculate_temperatures(x)
        
        # do softmax for each pred_len
        # x = self.softmax(x / temprature)
        # x = self.softmax(x)
        # x = self.sigmoid(x)
        
        # x is shape [batch, pred_len, output_embed_dim] sum along pred_len then collapse pred_len dimension
        x = torch.sum(x, dim=1)
        x = x.view(-1, len(self.label_encoders[-1].classes_))
        # print(x.shape)

        return x  
    
    def norm(self, x, dim =0, means= None , stdev=None):
        if means is not None :  
            return x * stdev + means
        else : 
            means = x.mean(dim, keepdim=True).detach()
            x = x - means
            stdev = torch.sqrt(torch.var(x, dim=dim, keepdim=True, unbiased=False)+ 1e-5).detach() 
            x /= stdev
            return x , means ,  stdev 
        
class Config:
    def __init__(self, label_encoders):
        self.seq_len = 16
        self.pred_len = 1
        self.patch_size = 0
        self.stride = 0
        self.d_model = 128
        self.method = 0
        # since classification, need to embed the data
        self.features = len(label_encoders) - 1
        self.label_encoders = label_encoders
        self.device = device
  
config = Config(label_encoders)      
model = PAttn(config, device).to(device)
print(model)

PAttn(
  (in_layer): Linear(in_features=32, out_features=128, bias=True)
  (lstm): LSTM(128, 128, num_layers=2, batch_first=True)
  (out_layer): Linear(in_features=128, out_features=16, bias=True)
  (fc_layers): ModuleList(
    (0-4): 5 x Linear(in_features=128, out_features=128, bias=True)
  )
  (softmax): Softmax(dim=2)
)


In [30]:
# split into X and y, y is the last column
y = df_raw['delta_out']
X = df_raw.drop(columns=['delta_out'])

# covert y to one hot encoding
y = pd.get_dummies(y)

# split into train and test
train_size = int(0.8 * len(X))
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# use a dataloader to load the data
from torch.utils.data import DataLoader, Dataset
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y, seq_len, prediction_len):
        self.X = X
        self.y = y
        self.seq_len = seq_len
        self.pred_len = prediction_len
        
    def __len__(self):
        return len(self.X) - self.seq_len - self.pred_len + 1
    
    def __getitem__(self, idx):
        return self.X[idx:idx+self.seq_len], self.y[idx+self.seq_len:idx+self.seq_len+self.pred_len]
    
train_dataset = TimeSeriesDataset(X_train.values, y_train.values, config.seq_len, config.pred_len)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)

test_dataset = TimeSeriesDataset(X_test.values, y_test.values, config.seq_len, config.pred_len)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [31]:
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

class EarlyStopping:
    def __init__(self, patience=7, verbose=False, delta=0):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.Inf
        self.delta = delta

    def __call__(self, val_loss, model, path):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model, path)
        elif score < self.best_score + self.delta:
            self.counter += 1
            print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model, path)
            self.counter = 0

    def save_checkpoint(self, val_loss, model, path):
        if self.verbose:
            print(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}).  Saving model ...')
        torch.save(model.state_dict(), path + '/' + 'checkpoint.pth')
        self.val_loss_min = val_loss
        
        
early_stopping = EarlyStopping(patience=5, verbose=True)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

In [32]:
for epoch in range(10):
    model.train()
    train_loss = 0
    for i, (X, y) in enumerate(train_loader):
        X, y = X.float().to(device), y.float().to(device)
        optimizer.zero_grad()
        output = model(X)
        # collapse both in dim 1
        # output = output.view(-1, output.size(-1))
        y = y.view(-1, y.size(-1))
        print(output.shape, y.shape)
        loss = criterion(output, y)
        print(f'{i}th loss: {loss.item()}')
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)
    
    model.eval()
    test_loss = 0
    with torch.no_grad():
        for i, (X, y) in enumerate(test_loader):
            X, y = X.float().to(device), y.float().to(device)
            output = model(X)
            y = y.view(-1, y.size(-1))
            loss = criterion(output, y)
            test_loss += loss.item()
        test_loss /= len(test_loader)
    
    print(f'Epoch {epoch+1}/{10}, Train Loss: {train_loss}, Test Loss: {test_loss}')
    early_stopping(test_loss, model, 'models')
    if early_stopping.early_stop:
        print('Early stopping')
        break
    scheduler.step()

torch.Size([32, 16]) torch.Size([32, 16])
0th loss: 2.772932529449463
torch.Size([32, 16]) torch.Size([32, 16])
1th loss: 2.772753953933716
torch.Size([32, 16]) torch.Size([32, 16])
2th loss: 2.772407054901123
torch.Size([32, 16]) torch.Size([32, 16])
3th loss: 2.7723565101623535
torch.Size([32, 16]) torch.Size([32, 16])
4th loss: 2.771716594696045
torch.Size([32, 16]) torch.Size([32, 16])
5th loss: 2.7711081504821777
torch.Size([32, 16]) torch.Size([32, 16])
6th loss: 2.7717864513397217
torch.Size([32, 16]) torch.Size([32, 16])
7th loss: 2.7701761722564697
torch.Size([32, 16]) torch.Size([32, 16])
8th loss: 2.7705230712890625
torch.Size([32, 16]) torch.Size([32, 16])
9th loss: 2.769361972808838
torch.Size([32, 16]) torch.Size([32, 16])
10th loss: 2.7709500789642334
torch.Size([32, 16]) torch.Size([32, 16])
11th loss: 2.767800807952881
torch.Size([32, 16]) torch.Size([32, 16])
12th loss: 2.7690584659576416
torch.Size([32, 16]) torch.Size([32, 16])
13th loss: 2.767063617706299
torch.Siz

In [34]:
# %%
# Test inference on a few samples from the test set
model.eval()
with torch.no_grad():
    for i, (X, y) in enumerate(test_loader):
        if i >= 5:  # Limit to 5 samples for demonstration
            break
        X, y = X.float().to(device), y.float().to(device)
        output = model(X)
        
        # Get the most certain predictions
        predicted_classes = torch.argmax(output, dim=1).cpu().numpy()
        true_classes = torch.argmax(y, dim=1).cpu().numpy()
        
        # Decode the predictions and true labels
        decoded_predictions = label_encoders[-1].inverse_transform(predicted_classes.flatten())
        decoded_true_labels = label_encoders[-1].inverse_transform(true_classes.flatten())
        
        print(f"Sample {i + 1}:")
        print(f"True Labels: {decoded_true_labels}")
        print(f"Predicted Labels: {decoded_predictions}")

Sample 1:
True Labels: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1